# Step 2: FDA Data Preprocessing

This notebook handles the preprocessing of FDA data:
1. Load preprocessed FDA file (fda_preprocessed.xls)
2. Clean text and parse dates
3. Select relevant columns
4. Save cleaned CSV

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import re
import os
from datetime import datetime

## 1. Load FDA Dataset

In [2]:
# Define file paths
FDA_DATA_PATH = '../data/pre-processed/fda/'
OUTPUT_PATH = '../data/pre-processed/fda/'

# Ensure output directory exists
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Load FDA preprocessed data
# Note: The .xls file is actually a CSV file (comma-separated) despite the extension
input_file = os.path.join(FDA_DATA_PATH, 'fda_preprocessed.xls')
print(f"Loading FDA data from {input_file}...")

# Read as CSV with comma separator and handle bad lines
fda_df = pd.read_csv(input_file, sep=',', on_bad_lines='warn', low_memory=False)
print(f"FDA dataset loaded successfully!")

print(f"\nFDA dataset shape: {fda_df.shape}")

Loading FDA data from ../data/pre-processed/fda/fda_preprocessed.xls...
FDA dataset loaded successfully!

FDA dataset shape: (26000, 17)


In [3]:
# Inspect the dataset
print("FDA Dataset Columns:")
print(fda_df.columns.tolist())
print("\nFirst 5 rows:")
fda_df.head()

FDA Dataset Columns:
['recall_id', 'reason', 'product', 'classification', 'status', 'distribution', 'recall_date', 'company', 'city', 'state', 'country', 'quantity', 'code_info', 'recall_year', 'recall_month', 'severity', 'class_number']

First 5 rows:


,recall_id,reason,product,classification,status,distribution,recall_date,company,city,state,country,quantity,code_info,recall_year,recall_month,severity,class_number
0,F-0276-2017,Recall initiated as a precautionary measure du...,"CytoDetox, Hydrolyzed Clinoptilolite Fragments...",Class II,Terminated,"FL, MI, MS, and OH.",2016-08-08,Pharmatech LLC,Davie,FL,United States,"1,990 bottles","UPC No. 632687615989; Lot No. 30661601, Exp. D...",2016.0,8.0,MEDIUM,II
1,F-0865-2017,"Mooncake products, manufactured and distribute...",Koi Palace Mini Moon Cake Single Box - Mini Oo...,Class II,Terminated,"CA, WA, OR.",2016-08-31,Magic Gourmet Trading Inc,Millbrae,CA,United States,"2 cases (1 pc/bx, 48bx/cs)","FG-M1MOT-UW Best by Nov 1, 2016.",2016.0,8.0,MEDIUM,II
2,F-0609-2015,Virginia State (VDACS) found Listeria monocyto...,Crema GuateLinda (Guatemalan Style Cream) in i...,Class I,Terminated,"FL, GA. NC, and TN",2014-10-10,"Oasis Brands, Inc",Miami,FL,United States,144 pieces,UPC 635349 000390 Best By dates: 07/01/14 thr...,2014.0,10.0,HIGH,I
3,F-1922-2012,FreshPoint South Florida is recalling sliced f...,Yellow Onion. Product is labeled in part FC Sa...,Class I,Terminated,Products were distributed in South Florida.,2012-07-27,FreshPoint South Florida,Pompano Beach,FL,United States,7 cases,Item # 302940.,2012.0,7.0,HIGH,I
4,F-0904-2020,Firm was notified by supplier that Organic Gro...,Pure Planet Organic Parasite Cleanse Net Wt. 1...,Class III,Terminated,"nationwide, Canada and Netherlands",2020-02-24,"Organic By Nature, Inc.",Rancho Dominguez,CA,United States,xx,Lot codes: 72746,2020.0,2.0,LOW,III


In [4]:
# Check data types
print("Data types:")
print(fda_df.dtypes)

Data types:
recall_id          object
reason             object
product            object
classification     object
status             object
distribution       object
recall_date        object
company            object
city               object
state              object
country            object
quantity           object
code_info          object
recall_year       float64
recall_month      float64
severity           object
class_number       object
dtype: object


## 2. Explore Data Quality

In [5]:
# Check missing values
print("Missing values per column:")
missing_counts = fda_df.isnull().sum()
missing_pct = (fda_df.isnull().sum() / len(fda_df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing Percentage': missing_pct.round(2)
})
print(missing_df[missing_df['Missing Count'] > 0])

Missing values per column:
              Missing Count  Missing Percentage
recall_id                 1                0.00
recall_date               1                0.00
state                   350                1.35
quantity               1660                6.38
code_info               322                1.24
recall_year               1                0.00
recall_month              1                0.00


In [6]:
# Check unique values for categorical columns
print("Unique values in key columns:")
for col in fda_df.columns:
    unique_count = fda_df[col].nunique()
    print(f"{col}: {unique_count} unique values")

Unique values in key columns:
recall_id: 25999 unique values
reason: 7472 unique values
product: 25860 unique values
classification: 3 unique values
status: 3 unique values
distribution: 5634 unique values
recall_date: 3178 unique values
company: 5016 unique values
city: 1940 unique values
state: 58 unique values
country: 31 unique values
quantity: 14773 unique values
code_info: 17294 unique values
recall_year: 18 unique values
recall_month: 12 unique values
severity: 3 unique values
class_number: 3 unique values


## 3. Clean Text Data

In [7]:
def clean_text(text):
    """
    Clean text by:
    - Converting to string if not null
    - Removing extra whitespace
    - Stripping leading/trailing whitespace
    - Converting to lowercase
    """
    if pd.isna(text):
        return ''
    
    text = str(text)
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with single space
    text = text.strip()  # Remove leading/trailing whitespace
    
    return text

# Apply text cleaning to string columns
text_columns = fda_df.select_dtypes(include=['object']).columns

print(f"Cleaning {len(text_columns)} text columns...")
for col in text_columns:
    fda_df[col] = fda_df[col].apply(clean_text)

print("Text cleaning completed!")

Cleaning 15 text columns...
Text cleaning completed!


## 4. Parse and Clean Dates

In [8]:
# Identify date columns (usually contain 'date' in column name)
date_columns = [col for col in fda_df.columns if 'date' in col.lower()]
print(f"Potential date columns: {date_columns}")

# Parse date columns
for col in date_columns:
    try:
        # Convert to datetime
        fda_df[col + '_parsed'] = pd.to_datetime(fda_df[col], errors='coerce')
        
        # Extract useful date components
        fda_df[col + '_year'] = fda_df[col + '_parsed'].dt.year
        fda_df[col + '_month'] = fda_df[col + '_parsed'].dt.month
        
        print(f"Parsed {col}:")
        print(f"  - Valid dates: {fda_df[col + '_parsed'].notna().sum()}")
        print(f"  - Invalid dates: {fda_df[col + '_parsed'].isna().sum()}")
    except Exception as e:
        print(f"Error parsing {col}: {e}")

Potential date columns: ['recall_date']
Parsed recall_date:
  - Valid dates: 25999
  - Invalid dates: 1


## 5. Select Relevant Columns

In [9]:
# Display all columns for selection
print("All available columns:")
for i, col in enumerate(fda_df.columns):
    print(f"{i+1}. {col}")

All available columns:
1. recall_id
2. reason
3. product
4. classification
5. status
6. distribution
7. recall_date
8. company
9. city
10. state
11. country
12. quantity
13. code_info
14. recall_year
15. recall_month
16. severity
17. class_number
18. recall_date_parsed
19. recall_date_year
20. recall_date_month


In [10]:
# Define relevant columns (adjust based on actual column names in your data)
# Common FDA recall columns include: product_description, reason_for_recall, company, date, etc.

# For now, keep all columns that are not mostly null (>50% missing)
relevant_columns = [col for col in fda_df.columns 
                   if fda_df[col].notna().sum() / len(fda_df) > 0.5]

print(f"Keeping {len(relevant_columns)} columns with >50% data:")
print(relevant_columns)

# Filter to relevant columns
fda_cleaned = fda_df[relevant_columns].copy()

Keeping 20 columns with >50% data:
['recall_id', 'reason', 'product', 'classification', 'status', 'distribution', 'recall_date', 'company', 'city', 'state', 'country', 'quantity', 'code_info', 'recall_year', 'recall_month', 'severity', 'class_number', 'recall_date_parsed', 'recall_date_year', 'recall_date_month']


## 6. Handle Missing Values

In [11]:
# Fill missing values appropriately
print("Handling missing values...")

# For text columns, fill with empty string
text_cols = fda_cleaned.select_dtypes(include=['object']).columns
for col in text_cols:
    fda_cleaned[col] = fda_cleaned[col].fillna('')

# For numeric columns, fill with 0 or -1 (depending on context)
numeric_cols = fda_cleaned.select_dtypes(include=['number']).columns
for col in numeric_cols:
    fda_cleaned[col] = fda_cleaned[col].fillna(-1)

print("\nMissing values after handling:")
print(fda_cleaned.isnull().sum())

Handling missing values...

Missing values after handling:
recall_id             0
reason                0
product               0
classification        0
status                0
distribution          0
recall_date           0
company               0
city                  0
state                 0
country               0
quantity              0
code_info             0
recall_year           0
recall_month          0
severity              0
class_number          0
recall_date_parsed    1
recall_date_year      0
recall_date_month     0
dtype: int64


## 7. Remove Duplicates

In [12]:
# Check for duplicates
print(f"Total rows before deduplication: {len(fda_cleaned)}")
duplicates = fda_cleaned.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

# Remove duplicates
fda_cleaned = fda_cleaned.drop_duplicates()
print(f"Total rows after deduplication: {len(fda_cleaned)}")

Total rows before deduplication: 26000
Duplicate rows: 0
Total rows after deduplication: 26000


## 8. Data Validation

In [13]:
# Final data quality check
print("Final Dataset Summary:")
print(f"Total rows: {len(fda_cleaned)}")
print(f"Total columns: {len(fda_cleaned.columns)}")
print("\nData types:")
print(fda_cleaned.dtypes)
print("\nSample data:")
fda_cleaned.head()

Final Dataset Summary:
Total rows: 26000
Total columns: 20

Data types:
recall_id                     object
reason                        object
product                       object
classification                object
status                        object
distribution                  object
recall_date                   object
company                       object
city                          object
state                         object
country                       object
quantity                      object
code_info                     object
recall_year                  float64
recall_month                 float64
severity                      object
class_number                  object
recall_date_parsed    datetime64[ns]
recall_date_year             float64
recall_date_month            float64
dtype: object

Sample data:


,recall_id,reason,product,classification,status,distribution,recall_date,company,city,state,country,quantity,code_info,recall_year,recall_month,severity,class_number,recall_date_parsed,recall_date_year,recall_date_month
0,F-0276-2017,Recall initiated as a precautionary measure du...,"CytoDetox, Hydrolyzed Clinoptilolite Fragments...",Class II,Terminated,"FL, MI, MS, and OH.",2016-08-08,Pharmatech LLC,Davie,FL,United States,"1,990 bottles","UPC No. 632687615989; Lot No. 30661601, Exp. D...",2016.0,8.0,MEDIUM,II,2016-08-08,2016.0,8.0
1,F-0865-2017,"Mooncake products, manufactured and distribute...",Koi Palace Mini Moon Cake Single Box - Mini Oo...,Class II,Terminated,"CA, WA, OR.",2016-08-31,Magic Gourmet Trading Inc,Millbrae,CA,United States,"2 cases (1 pc/bx, 48bx/cs)","FG-M1MOT-UW Best by Nov 1, 2016.",2016.0,8.0,MEDIUM,II,2016-08-31,2016.0,8.0
2,F-0609-2015,Virginia State (VDACS) found Listeria monocyto...,Crema GuateLinda (Guatemalan Style Cream) in i...,Class I,Terminated,"FL, GA. NC, and TN",2014-10-10,"Oasis Brands, Inc",Miami,FL,United States,144 pieces,UPC 635349 000390 Best By dates: 07/01/14 thru...,2014.0,10.0,HIGH,I,2014-10-10,2014.0,10.0
3,F-1922-2012,FreshPoint South Florida is recalling sliced f...,Yellow Onion. Product is labeled in part FC Sa...,Class I,Terminated,Products were distributed in South Florida.,2012-07-27,FreshPoint South Florida,Pompano Beach,FL,United States,7 cases,Item # 302940.,2012.0,7.0,HIGH,I,2012-07-27,2012.0,7.0
4,F-0904-2020,Firm was notified by supplier that Organic Gro...,Pure Planet Organic Parasite Cleanse Net Wt. 1...,Class III,Terminated,"nationwide, Canada and Netherlands",2020-02-24,"Organic By Nature, Inc.",Rancho Dominguez,CA,United States,xx,Lot codes: 72746,2020.0,2.0,LOW,III,2020-02-24,2020.0,2.0


In [14]:
# Statistical summary for numeric columns
print("Statistical Summary (numeric columns):")
fda_cleaned.describe()

Statistical Summary (numeric columns):


,recall_year,recall_month,recall_date_parsed,recall_date_year,recall_date_month
count,26000.000000,26000.000000,25999,26000.000000,26000.000000
mean,2017.532615,6.664462,2018-02-13 13:28:52.082002944,2017.532615,6.664462
min,-1.000000,-1.000000,2008-06-07 00:00:00,-1.000000,-1.000000
25%,2014.000000,4.000000,2014-12-24 00:00:00,2014.000000,4.000000
50%,2017.000000,7.000000,2017-05-15 00:00:00,2017.000000,7.000000
75%,2021.000000,10.000000,2021-07-17 00:00:00,2021.000000,10.000000
max,2025.000000,12.000000,2025-11-14 00:00:00,2025.000000,12.000000
std,13.108559,3.342189,NaN,13.108559,3.342189


## 9. Save Cleaned Dataset

In [15]:
# Save the cleaned dataset
output_file = os.path.join(OUTPUT_PATH, 'cleaned_fda_data.csv')

print(f"Saving cleaned data to {output_file}...")
fda_cleaned.to_csv(output_file, index=False)

# Verify file was saved
if os.path.exists(output_file):
    file_size = os.path.getsize(output_file) / (1024 * 1024)  # Convert to MB
    print(f"\nFile saved successfully!")
    print(f"File size: {file_size:.2f} MB")
else:
    print("Error: File was not saved!")

Saving cleaned data to ../data/pre-processed/fda/cleaned_fda_data.csv...

File saved successfully!
File size: 15.08 MB


In [16]:
# Summary
print("\n" + "="*50)
print("FDA Data Preprocessing Complete!")
print("="*50)
print(f"\nInput file: fda_preprocessed.xls")
print(f"  - Original records: {len(fda_df)}")
print(f"\nOutput file: cleaned_fda_data.csv")
print(f"  - Cleaned records: {len(fda_cleaned)}")
print(f"  - Columns kept: {len(fda_cleaned.columns)}")


FDA Data Preprocessing Complete!

Input file: fda_preprocessed.xls
  - Original records: 26000

Output file: cleaned_fda_data.csv
  - Cleaned records: 26000
  - Columns kept: 20
